In [1]:
from common import *

Function subs: 4
1st deriv subs: 20
2nd deriv subs: 100


This component is not suitable to compute using our formalism, just remember it: $R_{0101} = \frac{1}{4\tau_2^2} |\nabla \tau|^2$.

# $DP$ sector
Here we demonstrate the code for solving for the coefficients of writing $D_m P_n$ as linear combinations of the $R_{ambn}$

In [2]:
DP_mn_sol = decompose(func2sym(DP[m,n]), list(func2sym(Rd_ambn[m,n])), (m, n))
DP_mn_sol

[(\tau_{1}**2 + 2*I*\tau_{1}*\tau_{2} - \tau_{2}**2)/(2*\tau_{2}),
 (-\tau_{1} - I*\tau_{2})/(2*\tau_{2}),
 (-\tau_{1} - I*\tau_{2})/(2*\tau_{2}),
 1/(2*\tau_{2})]

We find that $D_m P_n = A_{mn} + iB_{mn}$, where

$$A_{mn} = \frac{\nabla_m \tau_2 \nabla_n \tau_2}{2\tau_2^2} - \frac{\nabla_m \nabla_n \tau_2}{2\tau_2} - \frac{\nabla_m \tau_1 \nabla_n \tau_1}{2\tau_2^2} = -\tau_2 \, R_{1m1n} + \tfrac{1}{2} g^{ab} R_{ambn}$$

$$B_{mn} = \frac{\nabla_m \nabla_n \tau_1}{2\tau_2} - \frac{\nabla_m \tau_1 \nabla_n \tau_2}{2\tau_2^2} - \frac{\nabla_m \tau_2 \nabla_n \tau_1}{2\tau_2^2} = \tau_1 \, R_{1m1n} - \tfrac{1}{2}(R_{1m2n} + R_{2m1n})$$

and correspondingly $D_m \bar{P}_n = A_{mn} - iB_{mn}$.

---

Note that alternatively, one can define
$$v^a = (\tau,; -1)$$

Then:

$$\boxed{D_m P_n = \frac{1}{2\tau_2}, v^a v^b, R_{ambn}}, \qquad \boxed{D_m \bar{P}n = \frac{1}{2\tau_2}, \bar{v}^a \bar{v}^b, R{ambn}}$$

where $v^a$ is a null vector of the complexified torus metric: $g_{ab} v^a v^b = 0$, with normalization $g_{ab} v^a \bar{v}^b = 2\tau_2$.
The inverse metric decomposes as:
$$g^{ab} = \frac{v^a \bar{v}^b + \bar{v}^a v^b}{2\tau_2}$$

In [3]:
# Reconstruct from solution and verify
reconstructed = sum(Rd_ambn[m,n][a,b] * DP_mn_sol[2*a +b] for a in range(2) for b in range(2))
reconstructed = sp.simplify(func2sym(reconstructed))

assert sp.simplify(reconstructed - func2sym(DP[m,n])) == 0

# $\bar{P}_m P_n$
$
\bar{P}_m P_n = \frac{1}{2} g^{ab} R_{ambn} - \frac{i}{2} R_{mn01}
$

In [4]:
PbP_mn_sol = decompose(func2sym( Pb[m] * P[n]), list(func2sym(Rd_ambn[m,n])), (m, n))
PbP_mn_sol

[(-\tau_{1}**2 - \tau_{2}**2)/(2*\tau_{2}),
 (\tau_{1} - I*\tau_{2})/(2*\tau_{2}),
 (\tau_{1} + I*\tau_{2})/(2*\tau_{2}),
 -1/(2*\tau_{2})]

In [5]:
# Reconstruct from solution and verify
reconstructed = sum(Rd_ambn[m,n][a,b] * PbP_mn_sol[2*a +b] for a in range(2) for b in range(2))
reconstructed = sp.simplify(func2sym(reconstructed))

assert sp.simplify(reconstructed - func2sym(Pb[m] * P[n])) == 0

# $P_m P_n$

To my knowledge this can not be written as a linear combination of 12d Riemann

In [6]:
PP_mn_sol = decompose(func2sym( P[m] * P[n]), list(func2sym(Rd_ambn[m,n])), (m, n))
PP_mn_sol

[]

The existing challenge is to write this as Riemann tensor components!

There it is. The obstruction is clear:

dt1m·dt1n and dt2m·dt2n both get coefficient -f11/(2τ₂) — they're always equal
But PP needs them with opposite signs: -1/(4τ₂²) vs +1/(4τ₂²)
This is the same obstruction as before, just in a different basis. The d2-cancellation constraint forces f00 = |τ|²f11 which ties everything to a single parameter f11, and that parameter multiplies dt1·dt1 and dt2·dt2 with the same coefficient.

Conclusion: P_m P_n cannot be written as c_{ab}(τ) times any index placement of R_{ambn} for m ≠ n. The issue is fundamental: after imposing the second-derivative cancellation, the remaining first-derivative freedom always produces (∂τ₁)² and (∂τ₂)² with the same sign, while P_m P_n needs opposite signs.

This worked for m = n because R^a{}_m{}^b{}_m has no second derivatives to begin with, so there's no constraint eating up the freedom.

# $P_m \bar{P}^m$

$P_m\bar{P}^m = R_{0101} = \frac{1}{4\tau_2^2} |\nabla \tau|^2$, computed by hand. One can also write this as a contraction of $\bar{P}_m P_n$ computed earlier.

# $D_m P_n D_r \bar{P}_s$
It can indeed be decomposed over $R_{ambn} R_{crds}$.

In [7]:
# D_m P_n * D_r Pb_s decomposed over R_{ambn} * R_{crds}
R_mn = list(func2sym(Rd_ambn[m,n]))
R_rs = list(func2sym(Rd_ambn[r,s]))
basis = [a * b for a in R_mn for b in R_rs]
target = sp.expand(func2sym(DP[m,n] * DPb[r,s]))
DPDPb_mnrs_sol = decompose(target, basis, (m,n), (r,s))

assert sp.simplify(target - sum(coef * basis[i] for i, coef in enumerate(DPDPb_mnrs_sol))) == 0

$\text{Re} D_m P_n D_r \bar{P}_s = \frac{1}{4} (g^{ac} g^{bd} - \epsilon^{ac} \epsilon^{bd}) R_{ambn} R_{crds}$

In [8]:
DPDPb_mnrs = real_parts(DPb_with_R[m,n] * DP_with_R[r,s])

RR_mnrs = 0
for a,c in iterprod(range(2), range(2)):
    ginv_ac = func2sym(ginv_func[a,c])
    epsilon_ac = func2sym(epsilon2_up[a,c])
    for b,d in iterprod(range(2), range(2)):
        ginv_bd = func2sym(ginv_func[b,d])  
        epsilon_bd = func2sym(epsilon2_up[b,d])  
        Riemann_ambn =Rd_ambn_sym[a, m, b, n]
        Riemann_crds = Rd_ambn_sym[c, r, d, s]
        RR_mnrs += sp.Rational(1,4) * (ginv_ac * ginv_bd - epsilon_ac * epsilon_bd) * Riemann_ambn * Riemann_crds

assert sp.simplify(DPDPb_mnrs - RR_mnrs) == 0

# $D_m \bar{P}^r D_n P_r$
just a contraction of the above

# $\text{Re} (D_m \bar{P}_s D_n \bar{P}_r P_p P^p + D_m P_s D_n P_r \bar{P}_p \bar{P}^p)$

In [9]:
# Target: Re(DPb_ms DPb_nr PP + DP_ms DP_nr PbPb)
target = real_parts(func2sym(
    DPb[m,s] * DPb[n,r] * P[p] * P[p] 
    + DP[m,s] * DP[n,r] * Pb[p] * Pb[p]
))
target = sp.expand(sp.simplify(target))

# Basis: R_{ambs} * R_{cndr} * R_{epfp}  (4 * 4 * 4 = 64 elements)
R_ms = list(func2sym(Rd_ambn[m,s]))
R_nr = list(func2sym(Rd_ambn[n,r]))
R_pp = list(func2sym(Rd_ambn[p,p]))
basis = [a*b*c for a in R_ms for b in R_nr for c in R_pp]

sol = decompose(target, basis, (m,s), (n,r), (p,p))


In [10]:
sol

[]

You're absolutely right. With raised indices, R^1{}_m{}^1{}m gives (-3(∂τ₁)² + (∂τ₂)²), which combined with g^{ab}R{ambm} = -((∂τ₁)² + (∂τ₂)²)/(2τ₂²) lets you extract both independent structures:

$$(\partial_m\tau_1)^2 - (\partial_m\tau_2)^2 = \tau_2^2, g^{ab}R_{ambm} - 2\tau_2^3, R^1{}{m}{}^1{}{m}$$

And similarly ∂_m τ₁ · ∂_m τ₂ can be extracted from R^0{}_m{}^1{}_m.

So my earlier analysis was wrong — I was only considering contractions with both indices down (g^{ab}, ε^{ab}), which forces equal-sign combinations. But raising indices with g^{ab} before contracting introduces the τ₁ cross terms from the off-diagonal metric, which breaks the degeneracy and gives access to the minus-sign combination.

This means P_m P_n and the full Re(DPb² PP + DP² PbPb) expression should be expressible purely in terms of 12d Riemann components after all. I owe you an apology for the earlier wrong conclusions.

# $DG_3$ sector
$|DG_3|^2 = (\nabla F_4)_a (\nabla F_4)^a$

In [5]:
nablaF4_sq = nablaF4_down[m].T * (ginv_func * nablaF4_down[n])
nablaF4_sq = func2sym(nablaF4_sq)[0]
nablaF4_sq = sp.expand(sp.simplify(nablaF4_sq))

DG3DG3b = DG3b[m] * DG3[n]
DG3DG3b = real_parts(func2sym(DG3DG3b))
DG3DG3b = func2sym(DG3DG3b)
DG3DG3b = sp.expand(sp.simplify(DG3DG3b))

assert sp.simplify(nablaF4_sq - DG3DG3b) == 0

$\text{Re} D_m P_n D_r \bar{G}_3 D_s \bar{G}_3 = \frac{1}{2} (g^{ac} g^{bd} - \epsilon^{ac} \epsilon^{bd}) R_{ambn} \nabla_r F_{4c} \nabla_s F_{4d}$

In [6]:
RnablaF4nablaF4_mnrs = 0
for a,c in iterprod(range(2), range(2)):
    ginv_ac = func2sym(ginv_func[a,c])
    epsilon_ac = func2sym(epsilon2_up[a,c])
    for b,d in iterprod(range(2), range(2)):
        ginv_bd = func2sym(ginv_func[b,d])  
        epsilon_bd = func2sym(epsilon2_up[b,d])  
        Riemann_ambn = R_ambn[m,n][a,b]
        nabla_r_F4_c = nablaF4_down[r][c]
        nabla_s_F4_d = nablaF4_down[s][d]
        RnablaF4nablaF4_mnrs += sp.Rational(1,2) * (ginv_ac * ginv_bd - epsilon_ac * epsilon_bd) * func2sym(Riemann_ambn * nabla_r_F4_c * nabla_s_F4_d)

DPDG3bDG3b = real_parts(func2sym(DP[m,n] * DG3b[r] * DG3b[s]))

assert sp.simplify(RnablaF4nablaF4_mnrs - DPDG3bDG3b) == 0


In [7]:
DPb_expr = sp.expand(sp.simplify(func2sym(DPb[(m, n)])))
R_expr = sp.expand(sp.simplify(func2sym(R_ambn[(m, n)])))

tau1_s, tau2_s = SYM_D0[tau1], SYM_D0[tau2]

c00, c01, c10, c11 = sp.symbols('c00 c01 c10 c11')
diff = sp.expand(DPb_expr - (c00*R_expr[0,0] + c01*R_expr[0,1] + c10*R_expr[1,0] + c11*R_expr[1,1]))

# collect one equation per derivative symbol
deriv_syms = diff.free_symbols - {tau1_s, tau2_s, c00, c01, c10, c11}
eqs = [sp.Eq(diff.coeff(ds), 0) for ds in deriv_syms]
# constant term (no derivative symbols)
eqs.append(sp.Eq(diff.subs([(ds, 0) for ds in deriv_syms]), 0))

sol = sp.solve(eqs, [c00, c01, c10, c11])


In [8]:
sol

{c00: (\tau_{1}**2 - 2*I*\tau_{1}*\tau_{2} - \tau_{2}**2)/(2*\tau_{2}),
 c01: (-\tau_{1} + I*\tau_{2})/(2*\tau_{2}),
 c10: (-\tau_{1} + I*\tau_{2})/(2*\tau_{2}),
 c11: 1/(2*\tau_{2})}

In [9]:
DPb_expr = sp.expand(sp.simplify(func2sym(DPb[(m, n)])))
R_expr = sp.expand(sp.simplify(func2sym(R_ambn[(m, n)])))

# unknowns
c00, c01, c10, c11 = sp.symbols('c00 c01 c10 c11')

eq = sp.Eq(DPb_expr,
           c00*R_expr[0,0] + c01*R_expr[0,1] + c10*R_expr[1,0] + c11*R_expr[1,1])

sol = sp.solve(eq, [c00, c01, c10, c11])


In [10]:
DPb_expr

-I*\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) - \nabla_{m} \nabla_{n} \tau_{2}/(2*\tau_{2}) - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2)

In [11]:
R_expr

Matrix([
[                                                                                                                                                                                                \nabla_{m} \nabla_{n} \tau_{2}/(2*\tau_{2}**2) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}/(4*\tau_{2}**3) - 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}/(4*\tau_{2}**3),                                                                                                                                                                          -\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) + \nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}/(4*\tau_{2}**3) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(4*\tau_{2}**2) + 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(4*\tau_{2}**2) - 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}*\tau_{1}/(4*\tau_{2}**3)],
[-\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) + \nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}/(2*\tau_{2}**2) 

In [12]:
sol

[((4*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{1}*\tau_{2}**2*c11 + 2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{2}**2*c01 + 2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{2}**2*c10 - 2*I*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{2}**2 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}**2*\tau_{2}*c11 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}*c01 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}*c10 + 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{2}**3*c11 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{2}**2 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}**2*c11 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}*c01 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}*c10 + 3*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{2}**2*c11 - 2*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{2} - 4*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}*c11 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{2}*c01 - 3*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{2}*c10 + 2*I*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{2} - 

In [13]:
DmPbn = sp.expand(sp.simplify(func2sym(DPb[m,n])))

In [14]:
DmPbn

-I*\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) - \nabla_{m} \nabla_{n} \tau_{2}/(2*\tau_{2}) - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2)